In [1]:
import pandas as pd

df = pd.read_csv("output/gold_dataset.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()


Rows: 3600
Columns: 29


,fiscal_year,fiscal_period,fiscal_period_name,cost_center_id,cost_center_name,department_name,gl_account_id,gl_account_name,pnl_category,actual_amount,...,variance_change_vs_prior,variance_trend_direction,rolling_3_period_variance,period_close_status,has_late_posting,late_posting_amount,late_posting_period_count,data_maturity_level,explanation_readiness,finance_signoff_flag
0,2024,1,FY24-P01,CC1001,Marketing Paid Media,Marketing,6100,Paid Advertising,Opex,306467.22,...,-15282.78,Stable,-15282.78,Hard Closed,False,0.0,0,Final,Ready,True
1,2024,1,FY24-P01,CC1001,Marketing Paid Media,Marketing,6200,Cloud Infrastructure,Opex,620304.27,...,-23195.73,Stable,-23195.73,Hard Closed,False,0.0,0,Final,Ready,True
2,2024,1,FY24-P01,CC1001,Marketing Paid Media,Marketing,6300,Professional Services,Opex,197696.09,...,4646.09,Stable,4646.09,Hard Closed,False,0.0,0,Final,Ready,True
3,2024,1,FY24-P01,CC1001,Marketing Paid Media,Marketing,6400,Travel & Entertainment,Opex,97398.48,...,873.48,Stable,873.48,Hard Closed,False,0.0,0,Final,Ready,True
4,2024,1,FY24-P01,CC1001,Marketing Paid Media,Marketing,6500,Software Licenses,Opex,226363.24,...,-9586.76,Stable,-9586.76,Hard Closed,False,0.0,0,Final,Ready,True


In [2]:
event = df[
    (df["fiscal_year"] == 2025) &
    (df["fiscal_period"].isin([11, 12])) &
    (df["gl_account_id"] == 6200) &
    (df["cost_center_id"].str.startswith("CC2"))
]

event[[
    "fiscal_period_name",
    "cost_center_name",
    "gl_account_name",
    "variance_pct",
    "variance_driver",
    "variance_direction"
]].sort_values("variance_pct", ascending=False)


,fiscal_period_name,cost_center_name,gl_account_name,variance_pct,variance_driver,variance_direction
3331,FY25-P11,Engineering Platform,Cloud Infrastructure,0.18,Cloud Expansion Program,Unfavorable
3341,FY25-P11,Engineering Product,Cloud Infrastructure,0.18,Cloud Expansion Program,Unfavorable
3351,FY25-P11,Engineering Infrastructure,Cloud Infrastructure,0.18,Cloud Expansion Program,Unfavorable
3481,FY25-P12,Engineering Platform,Cloud Infrastructure,0.18,Cloud Expansion Program,Unfavorable
3491,FY25-P12,Engineering Product,Cloud Infrastructure,0.18,Cloud Expansion Program,Unfavorable
3501,FY25-P12,Engineering Infrastructure,Cloud Infrastructure,0.18,Cloud Expansion Program,Unfavorable


In [3]:
df[df["variance_driver"] == "Cloud Expansion Program"][
    ["fiscal_period_name",
     "cost_center_name",
     "variance_amount"]
].groupby("fiscal_period_name").sum()


,cost_center_name,variance_amount
fiscal_period_name,,
FY25-P11,Engineering PlatformEngineering ProductEnginee...,518076.0
FY25-P12,Engineering PlatformEngineering ProductEnginee...,520603.2


In [4]:
df.sort_values("variance_amount", ascending=False).head(15)[
    ["fiscal_period_name",
     "cost_center_name",
     "gl_account_name",
     "variance_amount",
     "variance_driver"]
]


,fiscal_period_name,cost_center_name,gl_account_name,variance_amount,variance_driver
1541,FY24-P11,Engineering Product,Cloud Infrastructure,191129.49,One-Time
3481,FY25-P12,Engineering Platform,Cloud Infrastructure,173534.40,Cloud Expansion Program
3491,FY25-P12,Engineering Product,Cloud Infrastructure,173534.40,Cloud Expansion Program
3501,FY25-P12,Engineering Infrastructure,Cloud Infrastructure,173534.40,Cloud Expansion Program
3351,FY25-P11,Engineering Infrastructure,Cloud Infrastructure,172692.00,Cloud Expansion Program
3331,FY25-P11,Engineering Platform,Cloud Infrastructure,172692.00,Cloud Expansion Program
3341,FY25-P11,Engineering Product,Cloud Infrastructure,172692.00,Cloud Expansion Program
1681,FY24-P12,Engineering Platform,Cloud Infrastructure,166350.51,One-Time
3191,FY25-P10,Engineering Product,Cloud Infrastructure,161074.38,One-Time
2901,FY25-P08,Engineering Infrastructure,Cloud Infrastructure,154870.50,One-Time


In [5]:
def executive_summary_for_period(df, fiscal_year, fiscal_period):
    subset = df[
        (df["fiscal_year"] == fiscal_year) &
        (df["fiscal_period"] == fiscal_period)
    ]

    top = subset.sort_values("variance_amount", ascending=False).head(5)

    total_unfavorable = subset[subset["variance_amount"] > 0]["variance_amount"].sum()
    total_favorable = subset[subset["variance_amount"] < 0]["variance_amount"].sum()

    return {
        "period": f"{fiscal_year}-P{fiscal_period}",
        "top_drivers": top[[
            "cost_center_name",
            "gl_account_name",
            "variance_amount",
            "variance_driver"
        ]],
        "total_unfavorable": round(total_unfavorable, 2),
        "total_favorable": round(total_favorable, 2)
    }

summary = executive_summary_for_period(df, 2025, 11)
summary


{'period': '2025-P11',
 'top_drivers':                 cost_center_name       gl_account_name  variance_amount  \
 3341         Engineering Product  Cloud Infrastructure        172692.00   
 3331        Engineering Platform  Cloud Infrastructure        172692.00   
 3351  Engineering Infrastructure  Cloud Infrastructure        172692.00   
 3301        Marketing Paid Media  Cloud Infrastructure        101753.41   
 3441               IT Operations  Cloud Infrastructure         85290.23   
 
               variance_driver  
 3341  Cloud Expansion Program  
 3331  Cloud Expansion Program  
 3351  Cloud Expansion Program  
 3301                 One-Time  
 3441                 One-Time  ,
 'total_unfavorable': np.float64(1534762.35),
 'total_favorable': np.float64(-836199.07)}

In [6]:
def simulate_ai_explanation(summary):
    period = summary["period"]
    total_unf = summary["total_unfavorable"]
    total_fav = summary["total_favorable"]

    top = summary["top_drivers"]

    main_driver = top.iloc[0]

    explanation = f"""
For {period}, total unfavorable variance was approximately {total_unf:,.0f},
partially offset by favorable variances of {abs(total_fav):,.0f}.

The primary driver was {main_driver['gl_account_name']} within
{main_driver['cost_center_name']}, generating approximately
{main_driver['variance_amount']:,.0f} of unfavorable variance.

This variance is attributed to {main_driver['variance_driver']}.
The concentration of impact across Engineering cost centers
indicates a structured programmatic expansion rather than
isolated operational volatility.
"""
    return explanation.strip()

print(simulate_ai_explanation(summary))


For 2025-P11, total unfavorable variance was approximately 1,534,762,
partially offset by favorable variances of 836,199.

The primary driver was Cloud Infrastructure within
Engineering Product, generating approximately
172,692 of unfavorable variance.

This variance is attributed to Cloud Expansion Program.
The concentration of impact across Engineering cost centers
indicates a structured programmatic expansion rather than
isolated operational volatility.


In [8]:
def simulate_ai_explanation(summary):
    period = summary["period"]
    total_unf = float(summary["total_unfavorable"])
    total_fav = float(summary["total_favorable"])
    net = total_unf + total_fav

    top = summary["top_drivers"]
    main_driver = top.iloc[0]

    tone = "elevated" if net > 500000 else "moderate"

    explanation = f"""
For {period}, total unfavorable variance was {total_unf:,.0f},
partially offset by favorable movements of {abs(total_fav):,.0f},
resulting in a net impact of {net:,.0f}.

The primary driver was {main_driver['gl_account_name']} within
{main_driver['cost_center_name']}, contributing approximately
{main_driver['variance_amount']:,.0f} of unfavorable variance.

This movement is linked to {main_driver['variance_driver']}.
The concentration of impact across Engineering cost centers
suggests a coordinated expansion initiative rather than
isolated operational fluctuation.

Overall period risk level appears {tone} relative to baseline volatility.
"""
    return explanation.strip()


In [10]:
def simulate_ai_explanation(summary):
    period = summary["period"]
    total_unf = float(summary["total_unfavorable"])
    total_fav = float(summary["total_favorable"])
    net = total_unf + total_fav

    top = summary["top_drivers"]
    main_driver = top.iloc[0]

    tone = "elevated" if net > 500000 else "moderate"

    explanation = f"""
For {period}, total unfavorable variance was {total_unf:,.0f},
partially offset by favorable movements of {abs(total_fav):,.0f},
resulting in a net impact of {net:,.0f}.

The primary driver was {main_driver['gl_account_name']} within
{main_driver['cost_center_name']}, contributing approximately
{main_driver['variance_amount']:,.0f} of unfavorable variance.

This movement is linked to {main_driver['variance_driver']}.
The concentration of impact across Engineering cost centers
suggests a coordinated expansion initiative rather than
isolated operational fluctuation.

Overall period risk level appears {tone} relative to baseline volatility.
"""
    return explanation.strip()

print(simulate_ai_explanation(summary))



For 2025-P11, total unfavorable variance was 1,534,762,
partially offset by favorable movements of 836,199,
resulting in a net impact of 698,563.

The primary driver was Cloud Infrastructure within
Engineering Product, contributing approximately
172,692 of unfavorable variance.

This movement is linked to Cloud Expansion Program.
The concentration of impact across Engineering cost centers
suggests a coordinated expansion initiative rather than
isolated operational fluctuation.

Overall period risk level appears elevated relative to baseline volatility.
